# Prediction Table for Every Basket Row

This notebook loads the two final models of `03_ky_modelling_xgboost_lightgbm`, predicts **every basket row** of the four data sets (train, validation, test and unlisted), blends the predictions with the weight and the margins **frozen in `04_ky_ensemble_and_intervals`**, and saves one CSV with a `split` label on every row: `data/processed/regression/regression_predictions_all_splits.csv`.

**Rules kept here**
- **Nothing is fitted, tuned or re-estimated:** The blend weight and the margins are **read** from `ensemble_weight_and_margins.csv`.
- **No accuracy summary:** No error, accuracy or hit-rate summary is computed or shown for any split, because the test set is scored in one place only, which is `05_ky_test_evaluation`. The table carries `true_total_value` and a row-level 1 or 0 flag `within_70pct_range_after_ensemble`, but this notebook prints no rate.
- **Consistency checks only:** The notebook checks that the saved table agrees with the predictions already saved by `03_ky_modelling_xgboost_lightgbm`, `04_ky_ensemble_and_intervals` and `05_ky_test_evaluation`, which verifies consistency and not accuracy.
- **Untouched inputs:** The raw data and the earlier notebooks are not touched.

The per-state features for the classification model are not part of this table, because a basket row has no `Prediction_Time`. They are built in `03_ky_regression_feature_merging` (in `notebooks/classification/`).

## Columns of the saved table

| column | meaning |
|---|---|
| `Property_ID`, `Customer_ID` | home keys (`Customer_ID` comes from `property_split.csv`) |
| `split` | `train`, `validation`, `test` or `unlisted`. The label `unlisted` marks homes with 10 or more items that never form a classification state, and these are kept out of every fit |
| `classifier_split` | the label that the classification model gives that home (`not_in_classifier` for the unlisted homes, as written in `property_split.csv`) |
| `row_in_split` | row number inside its own parquet file, which allows a row to be traced back to `train_df`, `val_df`, `test_df` or `unlisted_df` |
| `n_documented` | basket size N (10, 20, 30 or 50) |
| `pred_is_in_sample` | `True` only for train rows, because the two models were fitted on them |
| `true_total_value` | the non-vehicle total of the home (the regression target, the same for every basket of a home) |
| `documented_value` | sum of the values of the N items in the basket (non-vehicle) |
| `xgb_pred`, `lgbm_pred` | the predictions of the two models in dollars (`expm1` of the log prediction) |
| `ensemble_pred` | `w x xgb_pred + (1 - w) x lgbm_pred` with the frozen `w` |
| `margin_70pct_confidence_after_ensemble` | the frozen relative margin of the 70% interval for this N, learned on the validation rows from the ensemble errors |
| `lower_70pct_confidence_after_ensemble` | `ensemble_pred x (1 - margin)` |
| `lower_70pct_confidence_after_ensemble_floored` | the lower bound, never below `documented_value` |
| `upper_70pct_confidence_after_ensemble` | `ensemble_pred x (1 + margin)` |
| `within_70pct_range_after_ensemble` | **1** if `true_total_value` lies between the floored lower bound and the upper bound, otherwise **0**. A basket never exceeds its home total, so using the raw lower bound gives the same 1 or 0 |

> **Caution on `within_70pct_range_after_ensemble`.** The flag is made with the **true** total, so it is an evaluation label that shows how the interval did. It is **not a feature that can be given to the classification model**, because it would leak the answer and a state has no true total at prediction time. This notebook only stores the 1 or 0 per row and prints **no hit rate for any split**, and the rates are reported in `04_ky_ensemble_and_intervals` and `05_ky_test_evaluation`.

In [1]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd, joblib

PROCESSED = Path("../../data/processed/regression")     # modelling tables from 02_ky_data_preparation
INTERIM = Path("../../data/interim/regression")
MODELS = Path("../../models/regression")
OUT_FILE = PROCESSED / "regression_predictions_all_splits.csv"
print("output file:", OUT_FILE)

output file: ..\..\data\processed\regression\regression_predictions_all_splits.csv


## 1. Load the frozen weight and margins and the two final models

Both come from earlier notebooks and are only **read** here.

In [2]:
frozen = pd.read_csv(MODELS / "ensemble_weight_and_margins.csv").set_index("item")["value"]      # written by 04_ky_ensemble_and_intervals
W_XGB = float(frozen["blend_weight_xgboost"])
MARGIN = {int(k.split("=")[1]): float(v) for k, v in frozen.items() if k.startswith("margin_N=")}
CONF = float(frozen["confidence_level"])
print(f"blend weight: XGBoost {W_XGB:.2f} / LightGBM {1 - W_XGB:.2f} | confidence level {CONF:.0%}")
print("frozen margin per N (relative, +/-):", {n: round(m, 4) for n, m in sorted(MARGIN.items())})

mx = joblib.load(MODELS / "xgb_log_model.joblib")
ml = joblib.load(MODELS / "lgbm_log_model.joblib")          # both fitted on train only (03_ky_modelling_xgboost_lightgbm) and trained on log1p(target)

blend weight: XGBoost 0.28 / LightGBM 0.72 | confidence level 70%
frozen margin per N (relative, +/-): {10: 0.8278, 20: 0.5111, 30: 0.4258, 50: 0.5903}


## 2. Predict every basket row of the four data sets

The feature columns are the same as in training, which means everything except `Property_ID` and `true_total_value`. The models are trained on the log of the target, so their predictions are converted back to dollars with `expm1`.

In [3]:
FILES = {"train": "train_df.parquet", "validation": "val_df.parquet", "test": "test_df.parquet", "unlisted": "unlisted_df.parquet"}
frames = {k: pd.read_parquet(PROCESSED / f) for k, f in FILES.items()}
FEATURE_COLS = [c for c in frames["train"].columns if c not in ("Property_ID", "true_total_value")]
for k, f in frames.items():
    assert [c for c in f.columns if c not in ("Property_ID", "true_total_value")] == FEATURE_COLS, k      # identical columns and order everywhere

keys = pd.read_csv(INTERIM / "property_split.csv")[["Property_ID", "Customer_ID", "classifier_split", "regression_split"]]
keys["classifier_split"] = keys["classifier_split"].fillna("none")

parts = []
for split, df in frames.items():
    t = pd.DataFrame({"Property_ID": df["Property_ID"].values, "split": split, "row_in_split": np.arange(len(df)),
                      "n_documented": df["n_documented"].values, "pred_is_in_sample": split == "train",
                      "true_total_value": df["true_total_value"].values, "documented_value": df["sum_value"].values})
    t["xgb_pred"] = np.expm1(mx.predict(df[FEATURE_COLS]))
    t["lgbm_pred"] = np.expm1(ml.predict(df[FEATURE_COLS]))
    parts.append(t)
table = pd.concat(parts, ignore_index=True)

table = table.merge(keys[["Property_ID", "Customer_ID", "classifier_split"]], on="Property_ID", how="left")
table["ensemble_pred"] = W_XGB * table["xgb_pred"] + (1 - W_XGB) * table["lgbm_pred"]
m = table["n_documented"].map(MARGIN)
assert m.notna().all(), "a basket size has no frozen margin"
table["margin_70pct_confidence_after_ensemble"] = m
table["lower_70pct_confidence_after_ensemble"] = table["ensemble_pred"] * (1 - m)
table["lower_70pct_confidence_after_ensemble_floored"] = np.maximum(table["lower_70pct_confidence_after_ensemble"], table["documented_value"])
table["upper_70pct_confidence_after_ensemble"] = table["ensemble_pred"] * (1 + m)
assert (table["true_total_value"] >= table["documented_value"] - 1e-9).all()          # a basket never exceeds the home total, so the floored and the raw lower bound give the same 1 / 0
table["within_70pct_range_after_ensemble"] = ((table["true_total_value"] >= table["lower_70pct_confidence_after_ensemble_floored"]) &
                                              (table["true_total_value"] <= table["upper_70pct_confidence_after_ensemble"])).astype(int)

COLS = ["Property_ID", "Customer_ID", "split", "classifier_split", "row_in_split", "n_documented", "pred_is_in_sample", "true_total_value", "documented_value",
        "xgb_pred", "lgbm_pred", "ensemble_pred", "margin_70pct_confidence_after_ensemble", "lower_70pct_confidence_after_ensemble",
        "lower_70pct_confidence_after_ensemble_floored", "upper_70pct_confidence_after_ensemble", "within_70pct_range_after_ensemble"]
table = table[COLS]
print(table.shape)

(1840, 17)


## 3. Checks (consistency and labels only, no accuracy)

1. **Counts:** The number of rows and homes per split is as expected.
2. **Labels:** Every home has exactly one label, and the label agrees with `property_split.csv`.
3. **Agreement with earlier notebooks:** The predictions agree with the ones that other notebooks already saved. The validation predictions equal `tuned_model_predictions.parquet` (`03_ky_modelling_xgboost_lightgbm`), the validation interval equals `ensemble_validation_intervals.parquet` (`04_ky_ensemble_and_intervals`), and the test predictions equal `test_ensemble_predictions.parquet` (`05_ky_test_evaluation`).

In [4]:
counts = table.groupby("split").agg(homes=("Property_ID", "nunique"), rows=("Property_ID", "size")).reindex(["train", "validation", "test", "unlisted"])
display(counts)
assert counts["rows"].tolist() == [1030, 230, 150, 430] and counts["homes"].tolist() == [81, 18, 13, 47]
assert len(table) == 1840 and table["Property_ID"].nunique() == 159

# (2) one label per home, same as the split file
lab = table.groupby("Property_ID")["split"].nunique()
assert (lab == 1).all(), "a home appears in two splits"
chk = table.drop_duplicates("Property_ID").merge(keys.rename(columns={"classifier_split": "classifier_split_file"}), on=["Property_ID", "Customer_ID"], how="left")
assert (chk["split"] == chk["regression_split"]).all(), "label differs from property_split.csv"
assert (chk["classifier_split"] == chk["classifier_split_file"]).all(), "classifier_split differs from the split file"
assert table["Customer_ID"].notna().all()
print("labels: one split per home and equal to property_split.csv")

# (3) agreement with predictions saved by earlier notebooks
sv = pd.read_parquet(INTERIM / "tuned_model_predictions.parquet")
sv = sv[sv["split"] == "val"].reset_index(drop=True)
v = table[table["split"] == "validation"].reset_index(drop=True)
assert np.allclose(v["xgb_pred"], sv["xgb_tuned_pred"]) and np.allclose(v["lgbm_pred"], sv["lgbm_tuned_pred"]), "validation predictions differ from 03_ky_modelling_xgboost_lightgbm"
iv = pd.read_parquet(INTERIM / "ensemble_validation_intervals.parquet").reset_index(drop=True)
assert np.allclose(v["ensemble_pred"], iv["ensemble_pred"]) and np.allclose(v["lower_70pct_confidence_after_ensemble_floored"], iv["lower_floor"]) and np.allclose(v["upper_70pct_confidence_after_ensemble"], iv["upper"]), "validation interval differs from 04_ky_ensemble_and_intervals"
ts = pd.read_parquet(INTERIM / "test_ensemble_predictions.parquet").reset_index(drop=True)
t = table[table["split"] == "test"].reset_index(drop=True)
assert (t["Property_ID"].values == ts["Property_ID"].values).all()
assert np.allclose(t["ensemble_pred"], ts["ens"]) and np.allclose(t["upper_70pct_confidence_after_ensemble"], ts["upper"]), "test predictions differ from 05_ky_test_evaluation"
assert (v["within_70pct_range_after_ensemble"].values == iv["hit_floor"].astype(int).values).all(), "validation 1 / 0 differs from 04_ky_ensemble_and_intervals"
assert (t["within_70pct_range_after_ensemble"].values == ts["inside_floor"].astype(int).values).all(), "test 1 / 0 differs from 05_ky_test_evaluation"
print("validation predictions and intervals equal the files saved by notebooks 03 and 04, test predictions and upper bounds equal the file saved by notebook 05, and the 1 / 0 column equals their hit flags (rates are not printed here)")

,homes,rows
split,,
train,81,1030
validation,18,230
test,13,150
unlisted,47,430


labels: one split per home and equal to property_split.csv
validation predictions and intervals equal the files saved by notebooks 03 and 04, test predictions and upper bounds equal the file saved by notebook 05, and the 1 / 0 column equals their hit flags (rates are not printed here)


## 4. What the table contains (counts and sizes only)

The cells below show the rows and homes per split and basket size. The prediction columns are described and not scored.

In [5]:
by_n = table.pivot_table(index="n_documented", columns="split", values="Property_ID", aggfunc="size", fill_value=0)[["train", "validation", "test", "unlisted"]]
by_n.loc["all"] = by_n.sum()
by_n.index.name = "rows by basket size N"
display(by_n)

below = table["ensemble_pred"] < table["documented_value"]
print("rows whose ensemble prediction is below the documented value of the basket (a clip at the documented value would change them):", int(below.sum()), "of", len(table))
print("by split:", table[below].groupby("split").size().to_dict(), "and homes:", table[below]["Property_ID"].nunique())
print("rows whose lower bound was raised by the floor:", int((table["lower_70pct_confidence_after_ensemble_floored"] > table["lower_70pct_confidence_after_ensemble"]).sum()))
display(table.groupby("split")[["xgb_pred", "lgbm_pred", "ensemble_pred", "true_total_value"]].describe().T.loc[(slice(None), ["min", "50%", "max"]), :].round(0))

split,train,validation,test,unlisted
rows by basket size N,,,,
10,405,90,65,235
20,285,65,45,110
30,210,45,25,60
50,130,30,15,25
all,1030,230,150,430


rows whose ensemble prediction is below the documented value of the basket (a clip at the documented value would change them): 17 of 1840
by split: {'train': 16, 'unlisted': 1} and homes: 4
rows whose lower bound was raised by the floor: 1162


split                     test      train  unlisted  validation
xgb_pred         min   17407.0    14500.0   17175.0     17146.0
                 50%   45170.0    40416.0   35704.0     40759.0
                 max  220146.0  1003758.0  169251.0    280542.0
lgbm_pred        min   17977.0    14869.0   17433.0     19045.0
                 50%   43002.0    40633.0   33855.0     41477.0
                 max  155925.0  1179707.0  135803.0    217935.0
ensemble_pred    min   17817.0    15105.0   17643.0     18615.0
                 50%   43134.0    40676.0   33973.0     41784.0
                 max  173907.0  1044831.0  133487.0    235465.0
true_total_value min    4065.0     4820.0    6046.0      2100.0
                 50%   49206.0    41532.0   24502.0     52847.0
                 max  138593.0  1349866.0  124084.0    360420.0

## 5. Save

One CSV holds all four data sets. The column `split` labels the rows, and `pred_is_in_sample` is `True` for train only.

In [6]:
table.to_csv(OUT_FILE, index=False)
back = pd.read_csv(OUT_FILE)
assert back.shape == table.shape and list(back.columns) == COLS
print("saved:", OUT_FILE, back.shape)
display(back.groupby("split").head(1).reset_index(drop=True).T)

saved: ..\..\data\processed\regression\regression_predictions_all_splits.csv (1840, 17)


,0,1,2,3
Property_ID,PRP-000011,PRP-000059,PRP-000040,PRP-000027
Customer_ID,CUS-00010,CUS-00051,CUS-00035,CUS-00024
split,train,validation,test,unlisted
classifier_split,train,validation,test,not_in_classifier
row_in_split,0,0,0,0
n_documented,10,10,10,10
pred_is_in_sample,True,False,False,False
true_total_value,41532.0,7966.0,24616.0,33178.0
documented_value,17169.0,7966.0,20116.0,16099.0
xgb_pred,42548.703,27594.094,38304.6,45221.75


## What the table contains and what was checked

- **The table is saved:** `data/processed/regression/regression_predictions_all_splits.csv` has 1,840 rows and 17 columns, with one row per basket and every row labelled with `split`. Train has 1,030 rows from 81 homes, validation 230 from 18, test 150 from 13 and unlisted 430 from 47. The flag `pred_is_in_sample` is `True` for the 1,030 train rows only, because validation, test and unlisted homes were never fitted on.
- **Nothing was refitted or retuned:** The weight (XGBoost 0.28 and LightGBM 0.72) and the margins (0.828, 0.511, 0.426 and 0.590 for N = 10, 20, 30 and 50) were read from the file saved by `04_ky_ensemble_and_intervals`, and the two models are the ones saved by `03_ky_modelling_xgboost_lightgbm`.
- **All consistency checks passed:** Every home has one label that equals `property_split.csv` and the classification model's label. The validation predictions and interval equal the files of `03_ky_modelling_xgboost_lightgbm` and `04_ky_ensemble_and_intervals`, and the test predictions and upper bounds equal the file of `05_ky_test_evaluation`. No error metric or hit rate was calculated for any split, because test scoring stays in `05_ky_test_evaluation`.
- **The flag `within_70pct_range_after_ensemble` (1 or 0):** It is 1 if the true total of the home lies inside the floored 70% range and 0 otherwise, and it equals the hit flags saved by `04_ky_ensemble_and_intervals` (validation) and `05_ky_test_evaluation` (test). It uses the true total, so it is an evaluation label and **must not be given to the classification model as a feature**. No hit rate is printed here.
- **Use of the columns:** The interval columns carry the 70% margin of the ensemble (`..._after_ensemble`). The floored lower bound is raised for 1,162 of the 1,840 rows, because the documented value of the basket already exceeds the raw lower bound.
- **Predictions below the documented value are rare at basket level:** 17 of the 1,840 rows (from 4 homes, 16 rows in train and 1 in unlisted, none in validation or test) have an ensemble prediction below the documented value of the basket. At classification states the documented value is the live sum over all items, so such cases can be more frequent there, and `03_ky_regression_feature_merging` (in `notebooks/classification/`) counts them.
- **Caution when reading train rows:** Their predictions are in-sample, so they should not be used to judge accuracy.